#### Simple Gen AI APP Using Langchain

In [30]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [31]:
# Clean text
def clean_text(text):
    lines = text.split("\n")  # Split into lines
    cleaned_lines = [line.strip() for line in lines if line.strip()]  # Remove empty lines and spaces
    cleaned_text = "\n".join(cleaned_lines)  # Join back
    return cleaned_text

In [32]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import TextLoader

loader=TextLoader('radius_QandA.txt')
docs=loader.load()
docs

[Document(metadata={'source': 'radius_QandA.txt'}, page_content='Frequently asked questions\nCommonly asked questions about best practices\nGeneral\nIs Kubernetes required to use Radius?\nCurrently yes. Although Radius is architected to run on any platform, today Kubernetes is the only hosting platform for Radius for the Radius control-plane and for containerized workloads. In the future, we plan to support other hosting platforms for serverless platforms.\n\nCan I incrementally adopt, or “try out” Radius?\nYes. The easiest way to add Radius to an existing application is through Radius annotations. Simply add the annotations to your existing Helm chart or Kubernetes YAML and you can use the Radius app graph, connections, and Recipes. Try the tutorial to learn more.\n\nDo I have to self-host Radius? Is there a managed service for Radius?\nOpen-source Radius requires that you self-host and run your own Radius instance in your Kubernetes cluster. In the future, we hope for providers to in

In [33]:
# Apply cleanup to each document
for doc in docs:
    doc.page_content = clean_text(doc.page_content)

In [34]:
docs

[Document(metadata={'source': 'radius_QandA.txt'}, page_content='Frequently asked questions\nCommonly asked questions about best practices\nGeneral\nIs Kubernetes required to use Radius?\nCurrently yes. Although Radius is architected to run on any platform, today Kubernetes is the only hosting platform for Radius for the Radius control-plane and for containerized workloads. In the future, we plan to support other hosting platforms for serverless platforms.\nCan I incrementally adopt, or “try out” Radius?\nYes. The easiest way to add Radius to an existing application is through Radius annotations. Simply add the annotations to your existing Helm chart or Kubernetes YAML and you can use the Radius app graph, connections, and Recipes. Try the tutorial to learn more.\nDo I have to self-host Radius? Is there a managed service for Radius?\nOpen-source Radius requires that you self-host and run your own Radius instance in your Kubernetes cluster. In the future, we hope for providers to includ

In [35]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [36]:
documents

[Document(metadata={'source': 'radius_QandA.txt'}, page_content='Frequently asked questions\nCommonly asked questions about best practices\nGeneral\nIs Kubernetes required to use Radius?\nCurrently yes. Although Radius is architected to run on any platform, today Kubernetes is the only hosting platform for Radius for the Radius control-plane and for containerized workloads. In the future, we plan to support other hosting platforms for serverless platforms.\nCan I incrementally adopt, or “try out” Radius?\nYes. The easiest way to add Radius to an existing application is through Radius annotations. Simply add the annotations to your existing Helm chart or Kubernetes YAML and you can use the Radius app graph, connections, and Recipes. Try the tutorial to learn more.\nDo I have to self-host Radius? Is there a managed service for Radius?\nOpen-source Radius requires that you self-host and run your own Radius instance in your Kubernetes cluster. In the future, we hope for providers to includ

In [67]:
import ollama
ollama.embed(model="nomic-embed-text", input=documents[0].page_content)

EmbedResponse(model='nomic-embed-text', created_at=None, done=None, done_reason=None, total_duration=428118100, load_duration=1588700, prompt_eval_count=211, prompt_eval_duration=None, eval_count=None, eval_duration=None, embeddings=[[0.010344164, 0.024880191, -0.1261447, -0.021799562, -0.009218314, -0.034396134, -0.015715772, 0.01391345, -0.004806024, -0.055887334, 0.032856334, 0.029739145, 0.06665853, 0.008434768, 0.017137282, -0.04230057, -0.023236861, -0.029628301, 0.017051691, 0.008485211, -0.04788522, -0.08765772, -0.012992189, -0.025464771, 0.020953028, 0.03791146, -0.005347853, 0.0051371427, -0.02416722, -0.0028364314, 0.026472526, -0.0073896996, 0.011493776, -0.008521083, 0.019347062, -0.04239724, 0.043626137, -0.020744413, -0.071983136, -0.0026854072, 0.035295688, -0.025813099, -0.035420947, -0.009373719, 0.06656434, -0.05114761, 0.09178867, 0.06869974, 0.08737537, -0.022108711, -0.030983424, -0.081140295, 0.011594364, -0.021107825, 0.033545334, 0.01608298, -0.017539136, 0.04

In [70]:
import faiss
import numpy as np
import ollama

# Generate embeddings using Ollama
embeddings = []
for doc in documents:
    response = ollama.embed(model="nomic-embed-text", input=doc.page_content)
    embedding = response.get('embeddings')  # Safely get 'embedding'
    if embedding is not None:
        embeddings.append(embedding)
    else:
        print(f"⚠️ Warning: No embedding found for '{doc.page_content}'")

# Convert the list of embeddings into a NumPy array of shape (N, D)
embeddings_array = np.array(embeddings, dtype=np.float32)

# Check the shape of embeddings
print(f"Embeddings shape before reshaping: {embeddings_array.shape}")

# Ensure it's 2D, and reshape if needed (e.g., from (N,) to (N, D))
if embeddings_array.ndim == 1:
    embeddings_array = embeddings_array.reshape(-1, len(embeddings_array[0]))  # Reshape if necessary

# Check shape after reshaping
print(f"Embeddings shape after reshaping: {embeddings_array.shape}")

# Get the embedding dimension (D)
dimension = embeddings_array.shape[1]  # Length of each embedding vector

# Create FAISS index (L2 distance - Euclidean)
index = faiss.IndexFlatL2(dimension)

# Add the embeddings to the FAISS index
index.add(embeddings_array)

print(f"Stored {embeddings_array.shape[0]} embeddings in FAISS.")

Embeddings shape before reshaping: (22, 1, 768)
Embeddings shape after reshaping: (22, 1, 768)


ValueError: too many values to unpack (expected 2)

In [71]:
from langchain_community.embeddings import OllamaEmbeddings
embeddings=(
    OllamaEmbeddings(model="gemma2")  ##by default it ues llama2
)

In [72]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

ValueError: Error raised by inference API HTTP code: 404, {"error":"model \"gemma2\" not found, try pulling it first"}

In [49]:
vectorstoredb

In [50]:
## Query From a vector db
query="How does Radius compare to Terraform"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'Similar to how you can define Azure resources in Bicep, you can define Radius resources in Bicep.\nRadius uses Bicep to add support for Radius resources. We previously used a temporary fork of Bicep, but have since deprecated it in favor of the main Bicep repo.\nTeams building or migrating applications on Radius can use Bicep to model their application and deploy to Kubernetes today, as well as future platforms, including serverless platforms.\nHow does Radius compare to Terraform?\nTerraform is a tool for building, changing, and versioning infrastructure safely and efficiently. Terraform is a great tool for deploying infrastructure, but doesn’t provide a way to model an entire application and the dependencies between services and infrastructure, or act as an abstraction layer for multiple cloud providers.'

In [53]:
## Query From a vector db
query="How does Radius compare to Waypoint"
result = vectorstoredb.similarity_search_with_score(query, k=1)

for doc, score in result:
    print(f"Score: {score}\nContent: {doc.page_content}\n")

Score: 0.2304379940032959
Content: Radius is a new project that takes what we learned from OAM and focuses more on the hosting platforms and the developer/operator experience, not just the spec. With Radius we’re providing an implementation that’s meant to be re-used and hosted in different scenarios. Right now we’re focused on Kubernetes. Radius is also taking a more holistic approach with Radius that’s bigger than just compute and networking. We’re doing more to work with existing tools like Terraform as well that are outside of Kubernetes. Lastly, Radius puts a lot of emphasis on the developer and operator handoff by building features such as Recipes and Environments that bind an application to different cloud providers and runtime platforms.
How does Radius compare to Waypoint?
HCP Waypoint is a HashiCorp-managed application deployment platform that simplifies the process of deploying applications into your infrastructure and helps you standardize your deployment process.



In [21]:
## Retrieval Chain, Document chain

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), config={'run_name': 'format_inputs'})
| ChatPromptTemplate(input_variables=['context'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'))])
| ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x0000023FB8A1DB10>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x0000023FB87D31F0>, model_name='gpt-4o', openai_api_key=SecretStr('**********'), openai_proxy='')
| StrOutputParser(), config={'run_name': 'stuff_documents_chain'})

In [23]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"LangSmith has two usage limits: total traces and extended",
    "context":[Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
})

'LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics tracked on their usage graph.'

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [24]:
### Input--->Retriever--->vectorstoredb

vectorstoredb

In [26]:
retriever=vectorstoredb.as_retriever()
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [27]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000023EF4D513F0>), config={'run_name': 'retrieve_documents'})
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), config={'run_name': 'format_inputs'})
            | ChatPromptTemplate(input_variables=['context'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'))])
            | ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x0000023FB8A1DB10>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x0000023

In [28]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"LangSmith has two usage limits: total traces and extended"})
response['answer']

'To set usage limits in LangSmith, navigate to **Settings -> Usage and Billing -> Usage configuration**. There, you will find a table at the bottom of the page that allows you to set usage limits per workspace. The two types of limits you can set are **total traces** and **extended retention traces**, each with an associated cost estimate.'

In [29]:

response

{'input': 'LangSmith has two usage limits: total traces and extended',
 'context': [Document(page_content='use usage limits to prevent future overspend.LangSmith has two usage limits: total traces and extended retention traces. These correspond to the two metrics we\'ve\nbeen tracking on our usage graph. We can use these in tandem to have granular control over spend.To set limits, we navigate back to Settings -> Usage and Billing -> Usage configuration. There is a table at the\nbottom of the page that lets you set usage limits per workspace. For each workspace, the two limits appear, along\nwith a cost estimate:Lets start by setting limits on our production usage, since that is where the majority of spend comes from.Setting a good total traces limit\u200bPicking the right "total traces" limit depends on the expected load of traces that you will send to LangSmith. You should\nclearly think about your assumptions before setting a limit.For example:Current Load: Our gen AI application is 

In [30]:
response['context']

[Document(page_content='use usage limits to prevent future overspend.LangSmith has two usage limits: total traces and extended retention traces. These correspond to the two metrics we\'ve\nbeen tracking on our usage graph. We can use these in tandem to have granular control over spend.To set limits, we navigate back to Settings -> Usage and Billing -> Usage configuration. There is a table at the\nbottom of the page that lets you set usage limits per workspace. For each workspace, the two limits appear, along\nwith a cost estimate:Lets start by setting limits on our production usage, since that is where the majority of spend comes from.Setting a good total traces limit\u200bPicking the right "total traces" limit depends on the expected load of traces that you will send to LangSmith. You should\nclearly think about your assumptions before setting a limit.For example:Current Load: Our gen AI application is called between 1.2-1.5 times per second, and each API request has a trace associate